In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [2]:
data_path = 'C://Users//Suchi_Kumari//GenAI_Capston//virtual-financial-advisor//data//virtual_financial_advisor_data.csv'
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(data_path)


In [9]:
from pyspark.sql.functions import col
df.filter(col("user_id") == "user_1").show()

+--------------+-------+----------+----------------+-------+--------------+----------------+--------------------+
|transaction_id|user_id|      date|        category| amount|payment_method|        merchant|         description|
+--------------+-------+----------+----------------+-------+--------------+----------------+--------------------+
|   txn_2551053| user_1|2024-08-24|      Healthcare| -324.3|          Cash|          Amazon|Healthcare paymen...|
|   txn_1351985| user_1|2024-03-05|   Entertainment| -25.47|          Cash|          Amazon|Entertainment pay...|
|   txn_6573264| user_1|2023-04-28|          Dining| -11.88|    Debit Card|         Walmart|Dining payment at...|
|   txn_4941513| user_1|2023-02-09|        Interest|1646.35|   Credit Card|Electric Company|Interest payment ...|
|   txn_2518233| user_1|2024-10-08|Savings Transfer|-312.42|          Cash|          School|Savings Transfer ...|
|   txn_2437584| user_1|2023-12-15|          Dining|-360.25|      Transfer|Electric Comp

In [10]:
from pyspark.sql.functions import col, abs, sum, date_format, year, month
df = df.withColumn("year_month", date_format(col("date"), 'yyyy-MM'))
df_credit = df.filter(col("amount") > 0).groupBy(col("user_id"), col("year_month")).agg(sum(col("amount")).alias("total_monthly_salary"))
df_debit = df.filter(col("amount") < 0).groupBy(col("user_id"), col("year_month")).agg(sum(abs(col("amount"))).alias("total_monthly_debit"))

df_user = df_credit.join(df_debit, "user_id", "inner")
df_user.orderBy(col("total_monthly_debit").desc()).show()

+-------+----------+--------------------+----------+-------------------+
|user_id|year_month|total_monthly_salary|year_month|total_monthly_debit|
+-------+----------+--------------------+----------+-------------------+
|user_18|   2023-03|             8944.09|   2024-02|  4926.089999999999|
|user_18|   2024-09|             4488.29|   2024-02|  4926.089999999999|
|user_18|   2024-05|             2212.47|   2024-02|  4926.089999999999|
|user_18|   2024-04|             2042.85|   2024-02|  4926.089999999999|
|user_18|   2023-04|             2559.45|   2024-02|  4926.089999999999|
|user_18|   2023-02|             1828.53|   2024-02|  4926.089999999999|
|user_18|   2023-11|             2957.53|   2024-02|  4926.089999999999|
|user_18|   2024-11|              2716.1|   2024-02|  4926.089999999999|
|user_18|   2024-10|             5949.97|   2024-02|  4926.089999999999|
|user_18|   2024-01|            22310.77|   2024-02|  4926.089999999999|
|user_18|   2023-01|   6852.290000000001|   2024-02

In [11]:
df_expense = df.select("user_id", "category", "amount", "year_month").groupBy(col("user_id"), col("category"), col("year_month")).agg((sum(col("amount"))).alias("cat_amount"))
df_expense.show()

+-------+----------------+----------+-------------------+
|user_id|        category|year_month|         cat_amount|
+-------+----------------+----------+-------------------+
|user_10|Savings Transfer|   2023-10|            -400.56|
|user_14|       Groceries|   2024-01|            -787.46|
| user_8|      Healthcare|   2023-07|             -20.33|
| user_1|       Transport|   2024-02|            -841.54|
| user_4|       Education|   2023-05|-377.46999999999997|
|user_10|      Healthcare|   2024-06|            -421.75|
| user_8|       Education|   2024-03|            -238.03|
|user_16|        Interest|   2024-04|            3140.01|
|user_15|          Dining|   2023-05|              -25.3|
|user_20|       Transport|   2024-08|            -495.92|
| user_7|       Groceries|   2023-10| -859.1800000000001|
|user_12|      Healthcare|   2024-08|             -18.07|
| user_4|       Utilities|   2024-01|            -463.07|
|user_10|            Rent|   2024-02|            -118.32|
|user_17|     

In [10]:
df_expense.filter(col("user_id") == 'user_10').show()

+-------+----------------+----------+------------------+
|user_id|        category|year_month|        cat_amount|
+-------+----------------+----------+------------------+
|user_10|Savings Transfer|   2023-10|           -400.56|
|user_10|      Healthcare|   2024-06|           -421.75|
|user_10|            Rent|   2024-02|           -118.32|
|user_10|       Utilities|   2024-01|           -153.51|
|user_10|   Entertainment|   2024-01|           -702.61|
|user_10|      Healthcare|   2024-04|           -310.24|
|user_10|   Entertainment|   2023-09|-772.0799999999999|
|user_10|          Dining|   2023-04|           -308.39|
|user_10|            Rent|   2023-09|           -762.95|
|user_10|       Transport|   2024-10|           -485.08|
|user_10|       Education|   2024-01|           -201.61|
|user_10|       Utilities|   2024-09|           -213.26|
|user_10|      Healthcare|   2023-07|           -197.11|
|user_10|       Utilities|   2023-08|            -50.51|
|user_10|       Transport|   20

In [12]:
from pyspark.sql.functions import sum

df_exp_all_col = (
    df_expense
    .groupBy("user_id", "year_month")
    .pivot("category")
    .agg(sum(col("cat_amount"))))
    
df_expense_pivot = df_exp_all_col.select("user_id","year_month",abs("Groceries"), abs("Rent"), abs("Utilities"), abs("Entertainment"), abs("Dining"), abs("Transport"), abs("Healthcare"), abs("Education"), abs("Savings Transfer"))


df_expense_pivot.show()

+-------+----------+------------------+---------+-----------------+------------------+------------------+------------------+-----------------+-----------------+---------------------+
|user_id|year_month|    abs(Groceries)|abs(Rent)|   abs(Utilities)|abs(Entertainment)|       abs(Dining)|    abs(Transport)|  abs(Healthcare)|   abs(Education)|abs(Savings Transfer)|
+-------+----------+------------------+---------+-----------------+------------------+------------------+------------------+-----------------+-----------------+---------------------+
|user_14|   2023-12|            534.69|     NULL|            648.9|             204.2|            235.67|             385.3|           132.57|            50.36|    943.6700000000001|
| user_7|   2024-01|            223.69|   902.99|             NULL|              NULL|            457.38|            646.14|          1222.49|             NULL|               145.49|
|user_12|   2024-02|              NULL|     NULL|           243.65|            511.25

In [17]:
df_exp_all_col.show()

+-------+----------+-------+-------------------+------------------+------------------+-------------------+------------------+--------+--------+-------+-------------------+-------------------+------------------+
|user_id|year_month|  Bonus|             Dining|         Education|     Entertainment|          Groceries|        Healthcare|Interest|    Rent| Salary|   Savings Transfer|          Transport|         Utilities|
+-------+----------+-------+-------------------+------------------+------------------+-------------------+------------------+--------+--------+-------+-------------------+-------------------+------------------+
|user_14|   2023-12|8898.64|            -235.67|            -50.36|            -204.2|            -534.69|           -132.57| 4966.26|    NULL|   NULL| -943.6700000000001|             -385.3|            -648.9|
| user_7|   2024-01|   NULL|            -457.38|              NULL|              NULL|            -223.69|          -1222.49| 2588.94| -902.99|   NULL|     

In [21]:
df_exp_all_col.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- Bonus: double (nullable = true)
 |-- Dining: double (nullable = true)
 |-- Education: double (nullable = true)
 |-- Entertainment: double (nullable = true)
 |-- Groceries: double (nullable = true)
 |-- Healthcare: double (nullable = true)
 |-- Interest: double (nullable = true)
 |-- Rent: double (nullable = true)
 |-- Salary: double (nullable = true)
 |-- Savings Transfer: double (nullable = true)
 |-- Transport: double (nullable = true)
 |-- Utilities: double (nullable = true)
 |-- total_income: double (nullable = true)



In [ ]:
from pyspark.sql.functions import lit, coalesce

df_exp_all_col = df_exp_all_col.withColumn(
    "total_income",
    coalesce(col("Salary"), lit(0.0)) +
    coalesce(col("Bonus"), lit(0.0)) +
    coalesce(col("Interest"), lit(0.0))
)
df_exp_all_col.show()

+-------+----------+-------+-------------------+------------------+------------------+-------------------+------------------+--------+--------+-------+-------------------+-------------------+------------------+------------------+
|user_id|year_month|  Bonus|             Dining|         Education|     Entertainment|          Groceries|        Healthcare|Interest|    Rent| Salary|   Savings Transfer|          Transport|         Utilities|      total_income|
+-------+----------+-------+-------------------+------------------+------------------+-------------------+------------------+--------+--------+-------+-------------------+-------------------+------------------+------------------+
|user_14|   2023-12|8898.64|            -235.67|            -50.36|            -204.2|            -534.69|           -132.57| 4966.26|    NULL|   NULL| -943.6700000000001|             -385.3|            -648.9|           13864.9|
| user_7|   2024-01|   NULL|            -457.38|              NULL|             

In [13]:
df_expense_user = df_expense_pivot.join(df_user, "user_id", "inner")

df_expense_user.show()

+-------+----------+------------------+---------+------------------+------------------+-----------+------------------+-----------------+-----------------+---------------------+----------+--------------------+----------+-------------------+
|user_id|year_month|    abs(Groceries)|abs(Rent)|    abs(Utilities)|abs(Entertainment)|abs(Dining)|    abs(Transport)|  abs(Healthcare)|   abs(Education)|abs(Savings Transfer)|year_month|total_monthly_salary|year_month|total_monthly_debit|
+-------+----------+------------------+---------+------------------+------------------+-----------+------------------+-----------------+-----------------+---------------------+----------+--------------------+----------+-------------------+
| user_7|   2024-03|              NULL|   322.72|             23.46|             47.97|     420.39|           1275.78|           702.72|           212.29|               134.03|   2024-03|             4530.72|   2024-01| 3598.1799999999994|
| user_7|   2024-05|            260.83| 

In [8]:
expense_cols = [
    "abs(Groceries)",
    "abs(Rent)",
    "abs(Utilities)",
    "abs(Entertainment)",
    "abs(Dining)",
    "abs(Transport)",
    "abs(Healthcare)",
    "abs(Education)",
    "abs(Savings Transfer)"
]

In [12]:
from pyspark.sql.functions import col, round

df_pct = df_expense_user

for c in expense_cols:
    df_pct = df_pct.withColumn(
        f"{c}_pct",
        round((col(c) / col("total_salary")) * 100, 2)
    )

df_pct.show()


+-------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+---------------------+------------------+------------------+------------------+-------------+------------------+----------------------+---------------+------------------+-------------------+------------------+-------------------------+
|user_id|    abs(Groceries)|        abs(Rent)|    abs(Utilities)|abs(Entertainment)|      abs(Dining)|    abs(Transport)|   abs(Healthcare)|    abs(Education)|abs(Savings Transfer)|      total_salary|       total_debit|abs(Groceries)_pct|abs(Rent)_pct|abs(Utilities)_pct|abs(Entertainment)_pct|abs(Dining)_pct|abs(Transport)_pct|abs(Healthcare)_pct|abs(Education)_pct|abs(Savings Transfer)_pct|
+-------+------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+---------------------+------------------+----------